In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Sem-4/ML DATASETSS/placement_predict_50k Dataset (1) (1).csv")

print("Dataset shape:", df.shape)
df.head()

Mounted at /content/drive
Dataset shape: (50000, 32)


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99


In [4]:
# ==============================
# List all column names
# ==============================

print("List all column names")
print(df.columns.tolist())

# ==============================
# Inspect data types
# ==============================
print("\nInspect data types")
df.info()


# ==============================
# Count missing values
# ==============================

print("\nCount missing values")
print(df.isnull().sum())


List all column names
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

Inspect data types
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   StudentID           50000 non-null  int64  
 1   Gender              50000 non-null  object 
 2   City                50000 non-null  object 
 3   CollegeTier         50000 non-null  object 
 4   Stream              50000 non-null  object 
 5   Specialisation      5000

In [5]:
# ==============================
# Check class balance
# ==============================

print("PlacementStatus:")
print(df["PlacementStatus"].value_counts())

print("\nCGPA_Tier:")
print(df["CGPA_Tier"].value_counts())

PlacementStatus:
PlacementStatus
1    32856
0    17144
Name: count, dtype: int64

CGPA_Tier:
CGPA_Tier
Low     16733
High    16681
Mid     16586
Name: count, dtype: int64


In [6]:
# ==============================
# Build general candidate feature set
# ==============================

exclude_cols = [
    "PlacementStatus",
    "CGPA_Tier",
    "Salary Package",
    "StudentID",
    "IsAnomaly"
]

X = df.drop(
    columns=exclude_cols,
    errors="ignore"
)

print("Number of features:", X.shape[1])
print(X.columns.tolist())


Number of features: 27
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular']


In [7]:
# ==============================
# Split numeric and categorical features
# ==============================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular']

Categorical features:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs']


In [8]:
# ==============================
# Numeric preprocessing pipeline
# ==============================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


In [9]:
# ==============================
# Categorical preprocessing pipeline
# ==============================

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)


In [10]:
# ==============================
# Combine preprocessors
# ==============================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


In [11]:
# ============================================================
# MODEL 1 - Binary Logistic Regression
# Predict PlacementStatus
# ============================================================

X_binary = df.drop(
    columns=[
        "PlacementStatus",
        "CGPA_Tier",
        "Salary Package",
        "StudentID",
        "IsAnomaly"
    ],
    errors="ignore"
)

y_binary = df["PlacementStatus"]


# Train/validation split

X_train_bin, X_val_bin, y_train_bin, y_val_bin = train_test_split(
    X_binary,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("Training samples:", X_train_bin.shape[0])
print("Validation samples:", X_val_bin.shape[0])


# Build Model 1 pipeline

binary_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)


# ==============================
# Reusable evaluation function
# ==============================

def evaluate_classifier(model, X_train, y_train, X_val, y_val):

    # Fit the model
    model.fit(X_train, y_train)

    # Predictions
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    # Accuracy
    train_accuracy = accuracy_score(
        y_train,
        train_pred
    )

    val_accuracy = accuracy_score(
        y_val,
        val_pred
    )

    print("=" * 70)
    print("MODEL EVALUATION")
    print("=" * 70)

    print(f"\nTrain Accuracy: {train_accuracy:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    print("\nClassification Report - Validation Data")
    print("-" * 70)
    print(classification_report(y_val, val_pred))

    print("\nConfusion Matrix - Validation Data")
    print("-" * 70)
    print(confusion_matrix(y_val, val_pred))

    return {
        "model": model,
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "train_predictions": train_pred,
        "val_predictions": val_pred
    }


# ==============================
# Train and evaluate Model 1
# ==============================

binary_result = evaluate_classifier(
    binary_lr,
    X_train_bin,
    y_train_bin,
    X_val_bin,
    y_val_bin
)


# ============================================================
# MODEL 2 - Multinomial Logistic Regression
# Predict CGPA_Tier
# ============================================================

X_multi = df.drop(
    columns=[
        "PlacementStatus",
        "CGPA_Tier",
        "Salary Package",
        "StudentID",
        "IsAnomaly"
    ],
    errors="ignore"
)

y_multi = df["CGPA_Tier"]


# Train/validation split

X_train_multi, X_val_multi, y_train_multi, y_val_multi = train_test_split(
    X_multi,
    y_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_multi
)

print("Training samples:", X_train_multi.shape[0])
print("Validation samples:", X_val_multi.shape[0])


# Build Model 2 pipeline
# Version-safe multinomial Logistic Regression

multi_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            random_state=42
        ))
    ]
)


Training samples: 40000
Validation samples: 10000
MODEL EVALUATION

Train Accuracy: 0.9257
Validation Accuracy: 0.9198

Classification Report - Validation Data
----------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.91      0.85      0.88      3429
           1       0.93      0.95      0.94      6571

    accuracy                           0.92     10000
   macro avg       0.92      0.90      0.91     10000
weighted avg       0.92      0.92      0.92     10000


Confusion Matrix - Validation Data
----------------------------------------------------------------------
[[2928  501]
 [ 301 6270]]
Training samples: 40000
Validation samples: 10000
